In [14]:
from IPython.display import display, Markdown
import json

from backend.permit_agent.agent_langchain import agent

In [15]:
def print_agent_trace(result):
    messages = result.get("messages", [])

    for msg in messages:
        # 1. Handle User Message
        if msg.type == "human":
            display(Markdown(f"### 👤 **User Request:**\n\n{msg.content}"))
            display(Markdown("---"))

        # 2. Handle AI Message (Reasoning or Tool Call)
        elif msg.type == "ai":
            # If the AI decided to call a tool
            if msg.tool_calls:
                for tool in msg.tool_calls:
                    args_formatted = json.dumps(tool['args'], indent=2)
                    display(Markdown(f"#### 🤖 **AI Decision (Tool Call):**\n"
                                     f"**Tool:** `{tool['name']}`\n\n"
                                     f"**Arguments:**\n```json\n{args_formatted}\n```"))
            # If this is the final answer (no tool calls, just content)
            elif msg.content:
                display(Markdown(f"### 🏁 **Final Answer:**\n\n{msg.content}"))

        # 3. Handle Tool Output (The raw data returned)
        elif msg.type == "tool":
            # We put this in a collapsible 'details' tag because tool outputs are often long
            clean_content = msg.content.replace('\n', '<br>')
            html_block = (
                f"<details><summary><strong>🛠️ Tool Output (Click to expand)</strong></summary>"
                f"<br><div style='background-color: #f4f4f4; padding: 10px; border-radius: 5px; font-family: monospace; font-size: 12px;'>"
                f"{clean_content}"
                f"</div></details>"
            )
            display(Markdown(html_block))


In [17]:
result = agent.invoke(
    {"messages": [
        {
            "role": "user",
            "content": "Sebutkan area pada PGN SOR 1 yang paling cepat akan kadaluwarsa dan kapan kadaluwarsanya ?"
        }
    ]}
)

print_agent_trace(result)

### 👤 **User Request:**

Sebutkan area pada PGN SOR 1 yang paling cepat akan kadaluwarsa dan kapan kadaluwarsanya ?

---

#### 🤖 **AI Decision (Tool Call):**
**Tool:** `get_list_documents_by_expiration_year`

**Arguments:**
```json
{
  "organization": "PGN SOR 1",
  "operator": "greater",
  "order_by": "earliest"
}
```

<details><summary><strong>🛠️ Tool Output (Click to expand)</strong></summary><br><div style='background-color: #f4f4f4; padding: 10px; border-radius: 5px; font-family: monospace; font-size: 12px;'>No documents found expiring in this year.</div></details>

### 🏁 **Final Answer:**

Berdasarkan konteks yang tersedia, tidak ditemukan dokumen izin pada area PGN SOR 1 yang akan segera kadaluwarsa. Jawaban ini didasarkan pada data yang ada dan mungkin belum lengkap jika ada dokumen yang belum terdata atau diperbarui.